импортируем все нужное и прочитаем данные. кроме текста и локации возьмем подкатегорию, цену, рейтинг и отзывы. дальше проверим, помогут ли они выбрать объявления, когда по тексту подходит сразу много вариантов

In [1]:
import gc
import re
import time
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from nltk.stem.snowball import RussianStemmer
from tqdm import tqdm
from catboost import CatBoostClassifier

started = time.time()
data_dir = Path('data')
search_cols = [
    'search_query', 'search_location_id', 'search_is_delivery_search',
    'search_infm_params_text', 'search_category',
]
item_cols = [
    'item_id', 'item_title_raw', 'item_description_raw', 'item_infm_params_text',
    'item_location_id', 'item_latitude', 'item_longitude',
    'item_microcat_id', 'item_price', 'item_rating', 'item_rating_reviews_count',
    'item_is_phone_hidden', 'item_is_message_forbidden',
]

train = pd.read_parquet(data_dir / 'train.parquet', columns=search_cols + item_cols)
queries = pd.read_parquet(data_dir / 'benchmark_queries.parquet')
benchmark_items = pd.read_parquet(data_dir / 'benchmark_items.parquet', columns=item_cols)
print('строк train:', len(train))
print('запросов benchmark:', len(queries))
print('объявлений benchmark:', len(benchmark_items))
print('совпадение локаций в train:', f'{(train.search_location_id == train.item_location_id).mean():.1%}')

строк train: 497673
запросов benchmark: 2452
объявлений benchmark: 189212
совпадение локаций в train: 83.1%


сначала подготовим текст: переведем его в нижний регистр, уберем знаки препинания и лишние пробелы. для поиска по словам будем обрезать окончания с помощью стемминга. так разные формы одного слова чаще будут совпадать при поиске

уберем повторяющиеся пары запроса и объявления. затем соберем объявления из train и benchmark в одну таблицу, чтобы искать среди них при проверке решения. каждый item_id оставим один раз. если объявление есть в обоих файлах, возьмем его данные из benchmark

еще нам понадобится примерный центр каждой локации. возьмем медиану широты и долготы ее объявлений. дальше от этой точки будем считать расстояние до объявления

In [2]:
def clean(text):
    text = str(text).lower().replace('\u0451', 'е').replace('\\n', ' ')
    return ' '.join(re.findall(r'[а-яa-z0-9]+', text))

stemmer = RussianStemmer()

@lru_cache(maxsize=150000)
def stem(word):
    return stemmer.stem(word)


def tokenize(text):
    return [stem(word) for word in clean(text).split() if len(word) > 1]


train['query_text'] = train.search_query.map(clean)
interactions = train[search_cols + ['query_text', 'item_id']].drop_duplicates().copy()
all_items = pd.concat([train[item_cols], benchmark_items], ignore_index=True)
all_items = all_items.drop_duplicates('item_id', keep='last').reset_index(drop=True)
del train
gc.collect()


all_items[['item_latitude', 'item_longitude']] = all_items[['item_latitude', 'item_longitude']].astype(float)
centers = all_items.groupby('item_location_id')[['item_latitude', 'item_longitude']].median()

known_queries = set(interactions.query_text)
print('новых текстов в benchmark:', f'{(~queries.search_query.map(clean).isin(known_queries)).mean():.1%}')
print('объявлений в корпусе для проверки:', len(all_items))
all_items['item_price'] = all_items.item_price.astype(float)
benchmark_items['item_price'] = benchmark_items.item_price.astype(float)
item_metadata = all_items[['item_id', 'item_location_id', 'item_microcat_id']].copy()

новых текстов в benchmark: 62.5%
объявлений в корпусе для проверки: 515895


разделим данные по текстам запросов. 5000 текстов возьмем для обучения модели отбора, 1800 для сравнения настроек и еще 1200 для последней проверки. для каждого текста случайно оставим один набор фильтров и локации. правильными ответами будут все объявления, выбранные при этих условиях

все строки с этими текстами уберем из истории поиска. иначе модель могла бы увидеть свой правильный ответ в запросах, добавленных к объявлению. тексты из прошлых экспериментов тоже не возвращаем в историю. для сравнения сохраним те же 1800 и 1200 запросов, что использовали раньше

In [3]:
rng = np.random.default_rng(42)
texts = interactions.query_text.unique()
first_texts = rng.choice(texts, 2400, replace=False)
remaining = texts[~pd.Index(texts).isin(first_texts)]
previous_test_texts = rng.choice(remaining, 1200, replace=False)
used_texts = set(first_texts) | set(previous_test_texts)

used_rows = interactions[interactions.query_text.isin(used_texts)]
used_queries = used_rows.groupby(search_cols, sort=False, dropna=False).agg(
    relevant=('item_id', lambda values: set(values)),
    query_text=('query_text', 'first'),
).reset_index()
used_queries = used_queries.sample(frac=1, random_state=42).drop_duplicates('query_text')

first_part = used_queries[used_queries.query_text.isin(first_texts)].sample(n=600, random_state=42)
second_part = used_queries[used_queries.query_text.isin(previous_test_texts)].sample(frac=1, random_state=42)
validation = pd.concat([second_part, first_part], ignore_index=True)

remaining = texts[~pd.Index(texts).isin(used_texts)]
test_texts = np.random.default_rng(314).choice(remaining, 1200, replace=False)
test_rows = interactions[interactions.query_text.isin(test_texts)]
test = test_rows.groupby(search_cols, sort=False, dropna=False).agg(
    relevant=('item_id', lambda values: set(values)),
    query_text=('query_text', 'first'),
).reset_index()
test = test.sample(frac=1, random_state=42).drop_duplicates('query_text')
test = test.sample(frac=1, random_state=43).reset_index(drop=True)

held_texts = used_texts | set(test_texts)
fit = interactions[~interactions.query_text.isin(held_texts)].copy()
assert not set(fit.query_text) & held_texts
assert not set(validation.query_text) & set(test.query_text)
print('обучающих пар:', len(fit))
print('для сравнения вариантов:', len(validation))
print('для итоговой проверки:', len(test))
rank_texts = np.random.default_rng(2026).choice(fit.query_text.unique(), 5000, replace=False)
rank_rows = fit[fit.query_text.isin(rank_texts)]
rank_queries = rank_rows.groupby(search_cols, sort=False, dropna=False).agg(
    relevant=('item_id', lambda values: set(values)), query_text=('query_text', 'first')
).reset_index().sample(frac=1, random_state=42).drop_duplicates('query_text').reset_index(drop=True)
fit = fit[~fit.query_text.isin(rank_texts)].copy()
assert not set(rank_queries.query_text) & set(fit.query_text)
assert not set(rank_queries.query_text) & held_texts
fit = fit.merge(item_metadata, on='item_id', how='left', validate='many_to_one')
interactions = interactions.merge(item_metadata, on='item_id', how='left', validate='many_to_one')
print('пар в истории поиска:', len(fit))
print('запросов для обучения отбора:', len(rank_queries))

обучающих пар: 441978
для сравнения вариантов: 1800
для итоговой проверки: 1200


пар в истории поиска: 414484
запросов для обучения отбора: 5000


соберем текст объявления из заголовка, параметров, описания и запросов из обучающей истории. заголовок повторим три раза, запросы из истории два раза. так слова из этих частей будут сильнее влиять на оценку

возьмем до восьми разных запросов на объявление, первые 400 символов параметров и первые 4500 символов описания. слова, которые встретились только один раз, тоже оставим. эти настройки выбрали в предыдущих экспериментах

по полученным текстам построим bm25. он оценивает совпадения слов, дает больший вес редким словам и учитывает длину текста

In [4]:
def build_word_index(items, history, description_length=4500, min_df=1, max_features=350000):
    known_queries = history.groupby('item_id').query_text.agg(
        lambda values: ' '.join(pd.unique(values)[:8])
    )
    documents = (items.item_title_raw.fillna('') + ' ') * 3
    documents += items.item_infm_params_text.fillna('').str[:400] + ' '
    documents += items.item_description_raw.fillna('').str[:description_length] + ' '
    documents += (items.item_id.map(known_queries).fillna('') + ' ') * 2

    vectorizer = CountVectorizer(
        tokenizer=tokenize, token_pattern=None, lowercase=False,
        min_df=min_df, max_df=0.9, max_features=max_features, dtype=np.float32,
    )
    counts = vectorizer.fit_transform(tqdm(documents, desc='слова', mininterval=20))
    lengths = np.asarray(counts.sum(axis=1)).ravel()
    document_frequency = np.bincount(counts.indices, minlength=counts.shape[1])
    idf = np.log1p((len(items) - document_frequency + 0.5) / (document_frequency + 0.5))

    k1 = 1.5
    b = 0.75
    length_norm = k1 * (1 - b + b * lengths / lengths.mean())
    entries = counts.tocoo()
    values = entries.data.copy()
    values *= (k1 + 1) / (entries.data + length_norm[entries.row])
    values *= idf[entries.col]
    word_index = sparse.csr_matrix(
        (values, (entries.col, entries.row)), shape=(counts.shape[1], len(items))
    )
    return vectorizer, word_index

к поиску по всему тексту добавим отдельные оценки по заголовку. одна будет учитывать совпадения слов, другая совпадения кусочков слов длиной от 3 до 5 символов. кусочки могут совпасть даже при опечатке или другом окончании

еще найдем похожие запросы в истории. для каждого из них известно, объявления каких подкатегорий выбирали пользователи. возьмем 30 самых похожих запросов и сложим их доли по подкатегориям. сходство возведем в четвертую степень, чтобы близкие формулировки влияли сильнее случайных совпадений

по этой же истории посчитаем связь локации поиска с локацией выбранного объявления. все это соберем в одной функции, потому что перед итоговым ответом поиск нужно будет пересобрать для benchmark_items

In [5]:
def build_search(items, history):
    words, word_index = build_word_index(items, history)
    chars = TfidfVectorizer(
        preprocessor=clean, analyzer='char_wb', ngram_range=(3, 5),
        min_df=3, max_features=180000, sublinear_tf=True, dtype=np.float32,
    )
    char_index = chars.fit_transform(items.item_title_raw.fillna('')).T.tocsr()

    title_counts = words.transform(items.item_title_raw.fillna(''))
    title_binary = title_counts.copy()
    title_binary.data[:] = 1
    lengths = np.asarray(title_counts.sum(axis=1)).ravel()
    df = np.bincount(title_counts.indices, minlength=title_counts.shape[1])
    idf = np.log1p((len(items) - df + 0.5) / (df + 0.5))
    length_norm = 1.5 * (0.25 + 0.75 * lengths / max(lengths.mean(), 1e-6))
    title_counts.data *= 2.5 / (title_counts.data + np.repeat(length_norm, np.diff(title_counts.indptr)))
    title_counts.data *= idf[title_counts.indices]
    title_index = title_counts.T.tocsr()

    query_categories = history.groupby(['query_text', 'item_microcat_id']).size().unstack(fill_value=0)
    query_categories = query_categories.div(query_categories.sum(axis=1), axis=0)
    query_vectorizer = TfidfVectorizer(
        preprocessor=clean, analyzer='char_wb', ngram_range=(3, 5), min_df=2,
        max_features=100000, sublinear_tf=True, dtype=np.float32,
    )
    query_text_index = query_vectorizer.fit_transform(query_categories.index).T.tocsr()
    query_category_matrix = sparse.csr_matrix(query_categories.to_numpy(dtype=np.float32))
    counts = history.groupby(['search_location_id', 'item_location_id']).size()
    location_probabilities = counts / counts.groupby(level=0).transform('sum')
    return {
        'items': items, 'fit': history, 'centers': centers,
        'word_vectorizer': words, 'word_index': word_index,
        'char_vectorizer': chars, 'char_index': char_index,
        'title_index': title_index, 'title_binary': title_binary.T.tocsr(),
        'location_probabilities': location_probabilities,
        'query_vectorizer': query_vectorizer, 'query_text_index': query_text_index,
        'query_category_matrix': query_category_matrix, 'category_ids': query_categories.columns.to_numpy(),
    }

search = build_search(all_items, fit)

слова:   0%|          | 0/515895 [00:00<?, ?it/s]

слова:  15%|█▍        | 76656/515895 [00:20<01:54, 3832.77it/s]

слова:  15%|█▍        | 76656/515895 [00:30<01:54, 3832.77it/s]

слова:  30%|██▉       | 154706/515895 [00:40<01:33, 3873.78it/s]

слова:  30%|██▉       | 154706/515895 [00:50<01:33, 3873.78it/s]

...

слова:  73%|███████▎  | 378091/515895 [01:40<00:37, 3718.81it/s]

слова:  73%|███████▎  | 378091/515895 [01:50<00:37, 3718.81it/s]

слова:  87%|████████▋ | 447587/515895 [02:00<00:18, 3635.84it/s]

слова:  87%|████████▋ | 447587/515895 [02:10<00:18, 3635.84it/s]

слова: 100%|██████████| 515895/515895 [02:19<00:00, 3689.26it/s]

теперь соберем кандидатов и признаки для модели. начнем с прежней оценки: 0.7 за совпадения слов, 0.3 за кусочки слов и 0.05 за текст фильтров. каждую из этих оценок сначала поделим на ее максимум для запроса, чтобы значения были от 0 до 1

оценку увеличим с учетом локации. совпадение локаций дает прибавку 7 к множителю. для других локаций добавим до 3 в зависимости от расстояния. еще добавим долю выборов этой локации из истории, взятую под корнем, с весом 3

возьмем 500 лучших объявлений по этой оценке. к ним добавим 200 по заголовку с учетом локации, 200 с учетом предполагаемой подкатегории и 100 только по тексту. повторы уберем. так у модели появятся варианты, которые прежний поиск сразу отсекал

для каждого кандидата сохраним все текстовые оценки, данные о локации, совместимость подкатегории с запросом и фильтром, рейтинг, отзывы и цену. также посчитаем долю слов запроса, найденных в заголовке

для обучения оставим первые 200 кандидатов, еще 200 случайных из остальных и все найденные правильные ответы. так не придется учиться на большом количестве похожих неподходящих объявлений. при проверке и подготовке answer.csv используем весь пул

In [6]:
def top_indices(scores, count):
    return np.argpartition(scores, -count)[-count:]


def make_features(cache, frame, name, training=False):
    items = cache['items']
    ids = items.item_id.to_numpy()
    query_words = cache['word_vectorizer'].transform(frame.search_query).tocsr()
    query_words.data[:] = 1
    query_filters = cache['word_vectorizer'].transform(frame.search_infm_params_text).tocsr()
    query_filters.data[:] = 1
    query_chars = cache['char_vectorizer'].transform(frame.search_query).tocsr()
    neighbor_chars = cache['query_vectorizer'].transform(frame.search_query).tocsr()
    locations = items.item_location_id.to_numpy()
    city_ids, city_codes = np.unique(locations, return_inverse=True)
    probabilities = cache['location_probabilities']
    known_locations = set(probabilities.index.get_level_values(0))
    centers = cache['centers']
    latitude = np.radians(items.item_latitude.to_numpy(dtype=np.float32))
    longitude = np.radians(items.item_longitude.to_numpy(dtype=np.float32))
    clean_titles = items.item_title_raw.fillna('').map(clean).to_numpy()
    microcat_codes = pd.Index(cache['category_ids']).get_indexer(items.item_microcat_id)
    popularity = cache['fit'].item_id.value_counts()
    filter_rows = cache['fit']
    filter_counts = filter_rows.groupby(['search_infm_params_text', 'item_microcat_id']).size()
    filter_probabilities = filter_counts / filter_counts.groupby(level=0).transform('sum')
    known_filters = set(filter_probabilities.index.get_level_values(0))
    microcats = items.item_microcat_id.to_numpy()
    del filter_rows
    fixed = np.column_stack([
        np.log1p(items.item_price.fillna(0).clip(lower=0)),
        items.item_rating.fillna(0),
        np.log1p(items.item_rating_reviews_count.fillna(0).clip(lower=0)),
        items.item_is_phone_hidden.fillna(False),
        items.item_is_message_forbidden.fillna(False),
        np.log1p(items.item_title_raw.fillna('').str.len()),
        np.log1p(items.item_description_raw.fillna('').str.len()),
        np.log1p(items.item_id.map(popularity).fillna(0)),
        items.item_price.lt(0),
        items.item_rating.isna(),
    ]).astype(np.float32)
    names = [
        'word', 'char', 'filters', 'title', 'title_coverage', 'word_raw', 'title_raw',
        'text_score', 'baseline_score', 'same_location', 'nearby', 'location_probability',
        'category_probability', 'category_similarity', 'neighbor_similarity', 'query_words',
        'query_chars', 'title_phrase', 'baseline_rank', 'filter_category', 'price', 'rating', 'reviews',
        'phone_hidden', 'message_forbidden', 'title_length', 'description_length', 'popularity', 'negative_price', 'missing_rating',
    ]
    blocks, labels, candidates, offsets = [], [], [], [0]
    rng = np.random.default_rng(42)
    baseline_recalls, pool_recalls = [], []
    for i, row in enumerate(tqdm(frame.itertuples(index=False), total=len(frame), desc=name, mininterval=20)):
        word_raw = (query_words[i] @ cache['word_index']).toarray().ravel()
        char = (query_chars[i] @ cache['char_index']).toarray().ravel()
        filters = (query_filters[i] @ cache['word_index']).toarray().ravel()
        title_raw = (query_words[i] @ cache['title_index']).toarray().ravel()
        word = word_raw / max(word_raw.max(), 1e-9)
        char /= max(char.max(), 1e-9)
        filters /= max(filters.max(), 1e-9)
        title = title_raw / max(title_raw.max(), 1e-9)
        coverage = (query_words[i] @ cache['title_binary']).toarray().ravel()
        coverage /= max(query_words[i].nnz, 1)

        same = locations == row.search_location_id
        nearby = np.zeros(len(items), dtype=np.float32)
        if row.search_location_id in centers.index:
            qlat, qlon = np.radians(centers.loc[row.search_location_id].to_numpy())
            a = np.sin((latitude - qlat) / 2) ** 2
            a += np.cos(qlat) * np.cos(latitude) * np.sin((longitude - qlon) / 2) ** 2
            distance = 6371 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
            nearby = np.nan_to_num(np.exp(-distance / 40), nan=0).astype(np.float32) * (~same)
        city_probability = np.zeros(len(items), dtype=np.float32)
        if row.search_location_id in known_locations:
            by_city = probabilities.loc[row.search_location_id].reindex(city_ids, fill_value=0).to_numpy(dtype=np.float32)
            city_probability = by_city[city_codes]

        similarities = (neighbor_chars[i] @ cache['query_text_index']).toarray().ravel()
        neighbors = top_indices(similarities, min(30, len(similarities)))
        weights = similarities[neighbors] ** 4
        category_distribution = np.asarray(weights @ cache['query_category_matrix'][neighbors]).ravel()
        category_distribution /= max(category_distribution.sum(), 1e-9)
        best_similarities = cache['query_category_matrix'][neighbors].copy()
        best_similarities.data[:] = 1
        best_similarities = best_similarities.multiply(similarities[neighbors, None]).max(axis=0).toarray().ravel()
        category_probability = np.where(microcat_codes >= 0, category_distribution[microcat_codes], 0)
        category_similarity = np.where(microcat_codes >= 0, best_similarities[microcat_codes], 0)
        text_score = 0.7 * word + 0.3 * char + 0.05 * filters
        geo = 1 + 7 * same + 3 * nearby + 3 * np.sqrt(city_probability)
        score = text_score * geo
        pool = np.unique(np.concatenate([
            top_indices(score, 500),
            top_indices((0.7 * title + 0.3 * char) * geo, 200),
            top_indices(score * (0.05 + category_probability), 200),
            top_indices(text_score, 100),
        ]))
        pool = pool[np.argsort(-score[pool], kind='stable')]
        relevant = row.relevant if hasattr(row, 'relevant') else set()
        y = np.isin(ids[pool], list(relevant))
        if relevant:
            baseline_recalls.append(len(set(ids[top_indices(score, 50)]) & relevant) / len(relevant))
            pool_recalls.append(y.sum() / len(relevant))
        rank = np.arange(len(pool))
        if training and len(pool) > 400:
            keep = np.unique(np.concatenate([
                np.arange(200), np.flatnonzero(y), rng.choice(np.arange(200, len(pool)), 200, replace=False),
            ]))
            pool, y, rank = pool[keep], y[keep], rank[keep]
        qtext = clean(row.search_query)
        filter_category = np.zeros(len(pool), dtype=np.float32)
        if row.search_infm_params_text and row.search_infm_params_text in known_filters:
            filter_category = filter_probabilities.loc[row.search_infm_params_text].reindex(
                microcats[pool], fill_value=0
            ).to_numpy(dtype=np.float32)
        dynamic = np.column_stack([
            word[pool], char[pool], filters[pool], title[pool], coverage[pool],
            word_raw[pool], title_raw[pool], text_score[pool], score[pool], same[pool],
            nearby[pool], city_probability[pool], category_probability[pool], category_similarity[pool],
            np.full(len(pool), similarities.max()), np.full(len(pool), query_words[i].nnz),
            np.full(len(pool), len(qtext)), [float(qtext in t) for t in clean_titles[pool]], np.log1p(rank), filter_category,
        ]).astype(np.float32)
        blocks.append(np.column_stack([dynamic, fixed[pool]]))
        labels.append(y)
        candidates.append(pool.astype(np.int32))
        offsets.append(offsets[-1] + len(pool))
    denominators = np.array([
        len(row.relevant) if hasattr(row, 'relevant') else 1
        for row in frame.itertuples(index=False)
    ])
    if baseline_recalls:
        print('исходный recall@50:', round(float(np.mean(baseline_recalls)), 4))
        print('recall всего пула:', round(float(np.mean(pool_recalls)), 4))
    return {
        'x': np.concatenate(blocks), 'y': np.concatenate(labels),
        'ids': np.concatenate(candidates), 'offsets': np.array(offsets),
        'denominators': denominators, 'names': names,
    }

подготовим примеры для обучения и сравнения настроек. метка 1 означает, что объявление есть среди выбранных пользователями, метка 0 что такого выбора в данных нет

проверим recall всего пула до обучения. если правильное объявление уже потерялось на этом шаге, модель отбора не сможет его вернуть

In [7]:
train_candidates = make_features(search, rank_queries, 'обучение', training=True)
validation_candidates = make_features(search, validation, 'сравнение')
print('размер обучающей таблицы:', train_candidates['x'].shape)

обучение:   0%|          | 0/5000 [00:00<?, ?it/s]

обучение:   7%|▋         | 358/5000 [00:20<04:19, 17.88it/s]

обучение:   7%|▋         | 358/5000 [00:38<04:19, 17.88it/s]

обучение:  15%|█▍        | 739/5000 [00:40<03:49, 18.54it/s]

обучение:  15%|█▍        | 739/5000 [00:58<03:49, 18.54it/s]

...

сравнение:  74%|███████▍  | 1337/1800 [01:00<00:20, 22.62it/s]

сравнение:  74%|███████▍  | 1337/1800 [01:11<00:20, 22.62it/s]

сравнение:  97%|█████████▋| 1753/1800 [01:20<00:02, 21.90it/s]

сравнение: 100%|██████████| 1800/1800 [01:22<00:00, 21.83it/s]

исходный recall@50: 0.7985
recall всего пула: 0.935
размер обучающей таблицы: (2000162, 30)


обучим catboost выбирать между кандидатами. он получит числовые признаки, которые мы посчитали выше. сам текст запроса и item_id в модель не передаем

правильных ответов среди кандидатов мало, поэтому при обучении дадим им вес 30. начнем с 600 деревьев глубины 6, а дальше сравним результат после 200, 400 и 600 деревьев

In [8]:
model = CatBoostClassifier(
    iterations=600, depth=6, learning_rate=0.06, l2_leaf_reg=6,
    loss_function='Logloss', class_weights=[1, 30], random_seed=42,
    thread_count=4, verbose=100, allow_writing_files=False,
)
model.fit(train_candidates['x'], train_candidates['y'])
del train_candidates
_ = gc.collect()

0:	learn: 0.5896377	total: 238ms	remaining: 2m 22s


100:	learn: 0.1034108	total: 17.3s	remaining: 1m 25s


200:	learn: 0.0971174	total: 33.1s	remaining: 1m 5s


300:	learn: 0.0923443	total: 46.5s	remaining: 46.2s


400:	learn: 0.0881332	total: 1m	remaining: 30s


500:	learn: 0.0846146	total: 1m 13s	remaining: 14.6s


599:	learn: 0.0813270	total: 1m 27s	remaining: 0us


проверим два способа выбрать итоговые 50 объявлений. первый использует только оценку модели. второй немного учитывает место объявления в исходном поиске

для смешивания возьмем обратные значения мест с добавлением 20 к знаменателю. так разница между соседними местами не будет слишком резкой. сравним вариант без смешивания и вариант с весом исходного поиска 0.15. выберем настройки по recall@50 на 1800 запросах

In [9]:
def model_scores(model, batch, trees, mix):
    scores = model.predict(batch['x'], prediction_type='RawFormulaVal', ntree_end=trees)
    if mix:
        for start, end in zip(batch['offsets'][:-1], batch['offsets'][1:]):
            order = np.argsort(-scores[start:end])
            ranks = np.empty(end - start, dtype=np.float32)
            ranks[order] = np.arange(end - start)
            scores[start:end] = (1 - mix) / (20 + ranks) + mix / (20 + np.arange(end - start))
    return scores


def measure_recall(scores, batch):
    values = []
    for i, (start, end) in enumerate(zip(batch['offsets'][:-1], batch['offsets'][1:])):
        top = np.argpartition(scores[start:end], -50)[-50:]
        values.append(batch['y'][start:end][top].sum() / batch['denominators'][i])
    return float(np.mean(values))


results = []
for trees in [200, 400, 600]:
    for mix in [0, 0.15]:
        scores = model_scores(model, validation_candidates, trees, mix)
        results.append({'trees': trees, 'mix': mix, 'recall@50': measure_recall(scores, validation_candidates)})
comparison = pd.DataFrame(results)
display(comparison)
best = comparison.loc[comparison['recall@50'].idxmax()]
best_trees = int(best.trees)
best_mix = float(best['mix'])
model.shrink(best_trees)
del validation_candidates
_ = gc.collect()

,trees,mix,recall@50
0,200,0.00,0.846667
1,200,0.15,0.845000
2,400,0.00,0.845000
3,400,0.15,0.845833
4,600,0.00,0.842315
5,600,0.15,0.841204


настройки выбрали. теперь проверим решение на 1200 отложенных запросах. рядом посчитаем результат прежней формулы на той же истории и среди тех же объявлений

по этой проверке настройки уже не меняем. она нужна, чтобы увидеть, сохранилось ли улучшение на других запросах

In [10]:
test_candidates = make_features(search, test, 'проверка')
test_scores = model_scores(model, test_candidates, best_trees, best_mix)
baseline_scores = test_candidates['x'][:, test_candidates['names'].index('baseline_score')]
display(pd.DataFrame({
    'вариант': ['прежняя формула', 'отбор моделью'],
    'recall@50': [measure_recall(baseline_scores, test_candidates), measure_recall(test_scores, test_candidates)],
}))
del test_candidates, test_scores, baseline_scores
_ = gc.collect()

проверка:   0%|          | 0/1200 [00:00<?, ?it/s]

проверка:  32%|███▏      | 379/1200 [00:20<00:43, 18.89it/s]

проверка:  32%|███▏      | 379/1200 [00:36<00:43, 18.89it/s]

проверка:  64%|██████▍   | 770/1200 [00:40<00:22, 19.27it/s]

проверка:  64%|██████▍   | 770/1200 [00:56<00:22, 19.27it/s]

проверка:  98%|█████████▊| 1175/1200 [01:00<00:01, 19.72it/s]

проверка: 100%|██████████| 1200/1200 [01:01<00:00, 19.57it/s]

исходный recall@50: 0.8
recall всего пула: 0.9306


,вариант,recall@50
0,прежняя формула,0.800000
1,отбор моделью,0.843889


локальная проверка закончена. пересоберем поиск только по benchmark_items, а для истории возьмем весь train. сведения о локациях и подкатегориях выбранных объявлений сохраним и для тех объявлений, которых нет в benchmark_items

модель отбора оставим с выбранными настройками. получим оценки для кандидатов каждого запроса и сохраним 50 лучших item_id в answer.csv

In [11]:
del search, all_items
stem.cache_clear()
gc.collect()
final_search = build_search(benchmark_items, interactions)
final_candidates = make_features(final_search, queries, 'ответ')
final_scores = model_scores(model, final_candidates, best_trees, best_mix)
item_ids = benchmark_items.item_id.to_numpy()
predictions = []
for start, end in zip(final_candidates['offsets'][:-1], final_candidates['offsets'][1:]):
    top = np.argpartition(final_scores[start:end], -50)[-50:]
    top = top[np.argsort(-final_scores[start:end][top])]
    positions = final_candidates['ids'][start:end][top]
    predictions.append(item_ids[positions].tolist())
answer = pd.DataFrame({
    'query_id': queries.query_id,
    'answer': [' '.join(found) for found in predictions],
})
answer.to_csv('answer.csv', index=False, encoding='utf-8')

слова:   0%|          | 0/189212 [00:00<?, ?it/s]

слова:  35%|███▌      | 66480/189212 [00:20<00:36, 3323.98it/s]

слова:  35%|███▌      | 66480/189212 [00:30<00:36, 3323.98it/s]

слова:  73%|███████▎  | 138195/189212 [00:40<00:14, 3477.96it/s]

слова:  73%|███████▎  | 138195/189212 [00:50<00:14, 3477.96it/s]

...

ответ:   0%|          | 0/2452 [00:00<?, ?it/s]

ответ:  43%|████▎     | 1065/2452 [00:20<00:26, 53.22it/s]

ответ:  43%|████▎     | 1065/2452 [00:32<00:26, 53.22it/s]

ответ:  91%|█████████ | 2235/2452 [00:40<00:03, 56.30it/s]

ответ: 100%|██████████| 2452/2452 [00:43<00:00, 56.06it/s]

прочитаем сохраненный answer.csv и проверим, что в нем ровно нужные query_id и каждый встречается один раз. в каждом ответе должно быть не больше 50 разных item_id, и все они должны быть в benchmark_items

еще проверим названия колонок и запись самих идентификаторов. при чтении явно оставим их строками, чтобы не потерять ведущие нули

In [12]:
saved = pd.read_csv('answer.csv', dtype=str, keep_default_na=False, encoding='utf-8')
assert saved.columns.tolist() == ['query_id', 'answer']
assert len(saved) == len(queries)
assert saved.query_id.is_unique
assert saved.query_id.str.len().eq(16).all()
assert set(saved.query_id) == set(queries.query_id)

allowed_ids = set(benchmark_items.item_id)
for value in saved.answer:
    item_ids = value.split(' ')
    assert 1 <= len(item_ids) <= 50
    assert len(item_ids) == len(set(item_ids))
    assert all(re.fullmatch(r'[0-9a-f]{16}', item_id) for item_id in item_ids)
    assert set(item_ids) <= allowed_ids

print('формат проверен:', len(saved), 'строки, по 50 уникальных item_id')
print('файл:', Path('answer.csv').resolve())
print('время выполнения:', round((time.time() - started) / 60, 1), 'мин')
saved.head(3)

формат проверен: 2452 строки, по 50 уникальных item_id
файл: /Applications/programming/vscode_projects/candidate-generation-for-a-service-category/answer.csv
время выполнения: 13.1 мин


,query_id,answer
0,70DfDUpwjxB4lzFd,b92ee8f432cec2d1 d722bcda1a555091 255fbeaf526a...
1,JTrdTaZJvSiLPkXj,367af128a9ea2a48 422d3ffdd5bbf626 cbeccbecb1fb...
2,LZCZNoVG4AFUkVRJ,d62074dd39caefee 168a9207e80b0be4 3f89b8062dc8...


для повторного запуска: python 3.11, папка data с тремя parquet файлами и следующие версии зависимостей
```bash
pip install numpy==2.4.6 pandas==3.0.5 scipy==1.17.1 scikit-learn==1.9.1 nltk==3.10.3 pyarrow==25.0.1 tqdm==4.70.1 catboost==1.2.10
```
использованные готовые методы: bm25, символьный tf idf из [scikit learn](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction), русский snowball из [nltk](https://www.nltk.org/api/nltk.stem.snowball.html), [catboost](https://catboost.ai/docs/en/concepts/python-reference_catboostclassifier)

все расчеты выполняются локально, внешние api не используются